# Input representation: one prepared file per raw trace file

The VeReMi-Extension data has one folder per attack scenario (`GridSybil_0709`, `DataReplaySybil_1416`, ...),
one sub-folder per simulation run, and inside it one `traceJSON-*.json` file per **receiving** vehicle.
Each line of such a file is either

* `"type": 2` — the receiver's own GPS fix (once per second), or
* `"type": 3` — a beacon the receiver heard from another vehicle (what that sender *claims* about itself).

This notebook keeps that structure. For every raw trace file it writes one prepared JSON file with the
**same name in the same folder tree** under the output folder. Nothing is de-duplicated: if the receiver
logged a message twice, both copies stay, because that is how the attack scenario was recorded.

For every received message we compute 13 numbers (the model input `x`):

| Feature | Meaning |
|---|---|
| `claimed_pos_x/y`, `claimed_vel_x/y`, `claimed_acl_x/y` | what the sender says about itself |
| `rx_pos_x/y`, `rx_vel_x/y` | the receiver's own latest GPS fix at the moment it heard the message |
| `range`, `bearing` | distance and direction (one angle in radians) from the receiver to the claimed position |
| `log_dtau` | log of the time since the receiver last heard the same sender pseudonym |

Messages are grouped into **links**: everything this receiver heard from one (sender pseudonym, true sender)
pair, in time order. **Every received message is kept** — the copies other receivers also logged, and the
first message of every link. That first message has no earlier message from the same pseudonym, so it has no
time gap: its `log_dtau` is stored as `null` (JSON's "no value"). Attackers that switch pseudonym every 1–2
messages produce many one-message links, so dropping them would remove the attack itself. Time of day, the `*_noise` fields and the ground-truth files are never used as inputs.
The true sender id is kept only as an identifier (for labels and for the vehicle-grouped split).

The raw data under `data/` is only read, never written.

In [1]:
from __future__ import annotations

import dataclasses
import json
import multiprocessing
import os
import pathlib
import re
import time
from typing import Any

import numpy as np

# Run from the repository root, so that `data/` and the output folder resolve the same way every time.
for _folder in [
    pathlib.Path.cwd(),
    *pathlib.Path.cwd().parents,
]:  # notebooks live in src/<folder>/; work from the repo root
    if (_folder / "CLAUDE.md").is_file():
        os.chdir(_folder)
        break
assert (
    pathlib.Path("CLAUDE.md").is_file() and pathlib.Path("data").is_dir()
), f"run this notebook from the repository root (now in {pathlib.Path.cwd()})"
print("working directory:", pathlib.Path.cwd())

working directory: /Users/bibekgupta/Downloads/projects/sybil-llm


## Settings

All settings live in one frozen dataclass so they can be stored next to the output. `out_dir` must not be
inside `data/` (that folder is read-only for this project).

In [2]:
@dataclasses.dataclass(frozen=True)
class PrepSettings:
    """Settings for turning raw VeReMi trace files into prepared files.

    Attributes:
        raw_dir: The raw VeReMi-Extension folder (read only).
        out_dir: Where the mirrored tree of prepared files and the index go.
        min_dtau_s: Smallest time gap (seconds) allowed before the log, so log(Δτ) stays finite.
        workers: Number of worker processes.
        decimals: Decimals kept for every float written to disk.
        seed: Seed for the vehicle split.
        train_frac: Share of sender vehicles in train.
        val_frac: Share of sender vehicles in val (test gets the rest).
        pretrain_val_frac: Share of the train vehicles kept aside for label-free checkpoint selection.
    """

    raw_dir: str = "data/VeReMi-Dataset"
    out_dir: str = "src/data/prepared_data"
    min_dtau_s: float = 1e-3
    workers: int = 10
    decimals: int = 4
    seed: int = 42
    train_frac: float = 0.70
    val_frac: float = 0.15
    pretrain_val_frac: float = 0.10

    def __post_init__(self) -> None:
        if self.min_dtau_s <= 0:
            raise ValueError(f"min_dtau_s must be > 0, got {self.min_dtau_s}")
        if self.workers < 1:
            raise ValueError(f"workers must be >= 1, got {self.workers}")
        for name in ("train_frac", "val_frac", "pretrain_val_frac"):
            if not 0.0 < getattr(self, name) < 1.0:
                raise ValueError(f"{name} must be between 0 and 1, got {getattr(self, name)}")
        if self.train_frac + self.val_frac >= 1.0:
            raise ValueError("train_frac + val_frac must leave room for test")
        out = pathlib.Path(self.out_dir).resolve()
        protected = pathlib.Path("data").resolve()
        if out == protected or protected in out.parents:
            raise ValueError(f"out_dir {self.out_dir} is inside the read-only data/ folder")


SETTINGS = PrepSettings()
SETTINGS

PrepSettings(raw_dir='data/VeReMi-Dataset', out_dir='src/data/prepared_data', min_dtau_s=0.001, workers=10, decimals=4, seed=42, train_frac=0.7, val_frac=0.15, pretrain_val_frac=0.1)

## The 13 features

The order below is the column order of every `x` row in every prepared file.

In [3]:
FEATURE_NAMES: tuple[str, ...] = (
    "claimed_pos_x",
    "claimed_pos_y",
    "rx_pos_x",
    "rx_pos_y",
    "claimed_vel_x",
    "claimed_vel_y",
    "rx_vel_x",
    "rx_vel_y",
    "claimed_acl_x",
    "claimed_acl_y",
    "range",
    "bearing",
    "log_dtau",
)
N_FEATURES = len(FEATURE_NAMES)
assert N_FEATURES == 13

# Attack code in a trace file name ("...-A<code>-...") -> class name. Only these codes exist locally.
ATTACK_NAMES: dict[int, str] = {
    0: "Benign",
    16: "GridSybil",
    17: "DataReplaySybil",
    18: "DoSRandomSybil",
    19: "DoSDisruptiveSybil",
}

## Reading one raw trace file

`TraceFile` knows where a file sits (scenario folder, run folder) and what its name says (receiver vehicle,
receiver pseudonym, receiver attack code). `RawTrace` reads its lines into three small arrays and checks the
format on the way: an unknown record type, a missing field, or a beacon whose receive time differs from its
send time stops the run with the file name and line number.

In [4]:
class RawFormatError(ValueError):
    """Raised when a raw trace file does not look like the documented VeReMi format."""


@dataclasses.dataclass(frozen=True)
class TraceFile:
    """One raw `traceJSON-*` file and what its path says.

    Attributes:
        path: The raw file.
        family: Attack family folder prefix, e.g. ``"GridSybil"``.
        group: Scenario group, ``"0709"`` or ``"1416"``.
        run_folder: Run folder name, e.g. ``"VeReMi_50400_54000_2022-9-11_19:13:55"``.
        time_window: Run folder name without its date, e.g. ``"VeReMi_50400_54000"``.
        receiver: Vehicle id of the receiver (first number in the file name).
        receiver_pseudo: The receiver's own pseudonym (second number).
        receiver_attack_code: The receiver's attack code (``A<code>``).
    """

    path: pathlib.Path
    family: str
    group: str
    run_folder: str
    time_window: str
    receiver: int
    receiver_pseudo: int
    receiver_attack_code: int

    _NAME = re.compile(r"^traceJSON-(\d+)-(\d+)-A(\d+)-(\d+)-(\d+)\.json$")
    _SCENARIO = re.compile(r"^([A-Za-z]+)_(\d{4})$")
    _RUN = re.compile(r"^(VeReMi_\d+_\d+)_")

    @classmethod
    def from_path(cls, path: str | pathlib.Path) -> "TraceFile":
        """Parses ``.../<Family>_<group>/<run folder>/traceJSON-...json``.

        Raises:
            RawFormatError: If the file or folder names do not follow the VeReMi pattern.
        """
        path = pathlib.Path(path)
        name = cls._NAME.match(path.name)
        scenario = cls._SCENARIO.match(path.parent.parent.name)
        run = cls._RUN.match(path.parent.name)
        if not (name and scenario and run):
            raise RawFormatError(f"not a VeReMi trace file path: {path}")
        return cls(
            path=path,
            family=scenario[1],
            group=scenario[2],
            run_folder=path.parent.name,
            time_window=run[1],
            receiver=int(name[1]),
            receiver_pseudo=int(name[2]),
            receiver_attack_code=int(name[3]),
        )

    @property
    def scenario(self) -> str:
        """Scenario folder name, e.g. ``"GridSybil_1416"``."""
        return f"{self.family}_{self.group}"

    @property
    def run(self) -> str:
        """``"<Family>_<group>/<run folder>"``."""
        return f"{self.scenario}/{self.run_folder}"

    def relative_path(self) -> pathlib.Path:
        """Path of this file relative to the raw root (and of its prepared copy relative to out_dir)."""
        return pathlib.Path(self.scenario, self.run_folder, self.path.name)


@dataclasses.dataclass(frozen=True)
class RawTrace:
    """The numbers one raw trace file contains, in file order.

    Attributes:
        own_gps: ``(n2, 5)`` float — rcv_time, pos x, pos y, vel x, vel y of the receiver's own fixes.
        heard: ``(n3, 7)`` float — rcv_time, claimed pos x/y, vel x/y, acl x/y of received messages.
        heard_ids: ``(n3, 3)`` int — true sender, sender pseudonym, message id.
    """

    own_gps: np.ndarray
    heard: np.ndarray
    heard_ids: np.ndarray

    @classmethod
    def read(cls, path: pathlib.Path) -> "RawTrace":
        """Reads one file line by line, keeping only the fields the features need.

        Raises:
            RawFormatError: On an unknown record type, a missing field, or ``rcvTime != sendTime``.
        """
        own_gps, heard, heard_ids = [], [], []
        with path.open("rb") as handle:
            for line_no, line in enumerate(handle, start=1):
                if not line.strip():
                    continue
                try:
                    rec = json.loads(line)
                    pos, spd = rec["pos"], rec["spd"]
                    if rec["type"] == 2:
                        own_gps.append((rec["rcvTime"], pos[0], pos[1], spd[0], spd[1]))
                    elif rec["type"] == 3:
                        if rec["sendTime"] != rec["rcvTime"]:
                            raise ValueError(f"rcvTime {rec['rcvTime']} != sendTime {rec['sendTime']}")
                        acl = rec["acl"]
                        heard.append((rec["rcvTime"], pos[0], pos[1], spd[0], spd[1], acl[0], acl[1]))
                        heard_ids.append((rec["sender"], rec["senderPseudo"], rec["messageID"]))
                    else:
                        raise ValueError(f"unknown record type {rec['type']!r}")
                except (KeyError, IndexError, TypeError, ValueError) as err:
                    raise RawFormatError(f"{path}:{line_no}: {err}") from err
        return cls(
            own_gps=np.asarray(own_gps, dtype=np.float64).reshape(-1, 5),
            heard=np.asarray(heard, dtype=np.float64).reshape(-1, 7),
            heard_ids=np.asarray(heard_ids, dtype=np.int64).reshape(-1, 3),
        )

## From one raw file to the receiver's view

`ReceiverView` does the work for one file:

* **Own GPS join.** Each received message gets the receiver's latest own fix with a time ≤ the message's
   receive time. Before the receiver's first fix there is none, so the nearest fix (its first one) is used.
* **Links.** Messages are grouped by (sender pseudonym, true sender) and put in time order (ties keep the
   file order). Pairing with the true sender keeps GridSybil's shared pseudonym 1 apart per ghost-sending car.
* **Time gap Δτ** to the previous message of the same link; `log_dtau = log(max(Δτ, min_dtau_s))`. The first
   message of a link has no Δτ, so its `log_dtau` is `null`; the message itself is kept. A message logged
   twice has Δτ = 0, which the floor keeps finite.
* **Range and bearing** of (claimed position − own position); bearing is one angle wrapped to (−π, π].

Two small pure functions do the geometry; everything else sits in the class.

In [5]:
def wrap_angle(angle: np.ndarray) -> np.ndarray:
    """Wraps angles (radians) to (−π, π]."""
    wrapped = np.mod(angle + np.pi, 2.0 * np.pi) - np.pi
    return np.where(wrapped == -np.pi, np.pi, wrapped)


def range_bearing(dx: np.ndarray, dy: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Distance and wrapped direction of the vector (dx, dy)."""
    return np.hypot(dx, dy), wrap_angle(np.arctan2(dy, dx))


@dataclasses.dataclass(frozen=True)
class Link:
    """Everything one receiver kept from one (sender pseudonym, true sender) pair, in time order.

    Attributes:
        sender_pseudo: The pseudonym the messages carried.
        sender: The true sending vehicle (identifier only, never an input).
        message_id: Message ids, one per kept message.
        rcv_time: Receive times (seconds), one per kept message.
        x: ``(n, 13)`` features, one row per kept message.
    """

    sender_pseudo: int
    sender: int
    message_id: np.ndarray
    rcv_time: np.ndarray
    x: np.ndarray


class ReceiverView:
    """Builds the links of one receiver from its raw trace.

    Args:
        settings: The preparation settings (uses ``min_dtau_s``).
    """

    def __init__(self, settings: PrepSettings) -> None:
        self._min_dtau_s = settings.min_dtau_s

    def build(self, trace: RawTrace) -> list[Link]:
        """Returns every link (all of its messages), ordered by sender pseudonym, then sender.

        Raises:
            RawFormatError: If the file has received messages but no own GPS fix.
        """
        if len(trace.heard) == 0:
            return []
        if len(trace.own_gps) == 0:
            raise RawFormatError("received messages but no own GPS fix")
        rx = self._own_fix_at(trace.own_gps, trace.heard[:, 0])
        sender, pseudo = trace.heard_ids[:, 0], trace.heard_ids[:, 1]
        # lexsort is stable: order by pseudonym, then sender, then time; equal times keep file order.
        order = np.lexsort((trace.heard[:, 0], sender, pseudo))
        heard, ids, rx = trace.heard[order], trace.heard_ids[order], rx[order]
        same_link = (ids[1:, 0] == ids[:-1, 0]) & (ids[1:, 1] == ids[:-1, 1])
        starts = np.flatnonzero(np.concatenate(([True], ~same_link)))
        ends = np.append(starts[1:], len(heard))
        links = []
        for start, end in zip(starts, ends):
            links.append(self._link(heard[start:end], ids[start:end], rx[start:end]))
        return links

    @staticmethod
    def _own_fix_at(own_gps: np.ndarray, times: np.ndarray) -> np.ndarray:
        """The receiver's latest own fix with time ≤ each time (its first fix if none is earlier)."""
        fixes = own_gps[np.argsort(own_gps[:, 0], kind="stable")]
        index = np.searchsorted(fixes[:, 0], times, side="right") - 1
        return fixes[np.maximum(index, 0), 1:5]

    def _link(self, heard: np.ndarray, ids: np.ndarray, rx: np.ndarray) -> Link:
        """Features of every message of one link; the first message's log_dtau is NaN (no earlier message)."""
        dtau = np.concatenate(([np.nan], np.diff(heard[:, 0])))
        claimed_pos, claimed_vel, claimed_acl = heard[:, 1:3], heard[:, 3:5], heard[:, 5:7]
        rx_pos, rx_vel = rx[:, 0:2], rx[:, 2:4]
        distance, bearing = range_bearing(claimed_pos[:, 0] - rx_pos[:, 0], claimed_pos[:, 1] - rx_pos[:, 1])
        x = np.column_stack(
            (
                claimed_pos,
                rx_pos,
                claimed_vel,
                rx_vel,
                claimed_acl,
                distance,
                bearing,
                np.log(np.maximum(dtau, self._min_dtau_s)),
            )
        )
        return Link(
            sender_pseudo=int(ids[0, 1]), sender=int(ids[0, 0]), message_id=ids[:, 2], rcv_time=heard[:, 0], x=x
        )

## Writing one prepared file and reporting on it

`TraceFilePreparer` ties it together for one raw file: read, build the links, round floats, write the
columnar JSON to the mirrored path, and return a small summary. The summary carries everything the index needs
later (counts, link lengths with the sender vehicle, and per-sender-vehicle partial sums n / sum / sum of
squares of the 13 features), so no prepared file has to be read again.

In [6]:
class TraceFilePreparer:
    """Turns one raw trace file into one prepared file plus a summary.

    Args:
        settings: The preparation settings.
    """

    def __init__(self, settings: PrepSettings) -> None:
        self._settings = settings
        self._view = ReceiverView(settings)
        self._raw_root = pathlib.Path(settings.raw_dir)
        self._out_root = pathlib.Path(settings.out_dir)

    def prepare(self, path: str | pathlib.Path) -> dict[str, Any]:
        """Prepares ``path`` and returns its summary.

        Raises:
            RawFormatError: If the raw file is malformed.
            ValueError: If a feature is not finite.
        """
        meta = TraceFile.from_path(path)
        trace = RawTrace.read(meta.path)
        links = self._view.build(trace)
        decimals = self._settings.decimals
        rounded = [np.round(link.x, decimals) for link in links]
        for link, x in zip(links, rounded):
            # Only the first log_dtau of a link may be missing; every other value must be a real number.
            if not (np.isfinite(x[:, :-1]).all() and np.isfinite(x[1:, -1]).all()):
                raise ValueError(f"non-finite feature in {meta.path} (sender {link.sender})")
        n_kept = sum(len(x) for x in rounded)
        n_type3 = len(trace.heard)
        payload = {
            "receiver": meta.receiver,
            "receiver_pseudo": meta.receiver_pseudo,
            "receiver_attack_code": meta.receiver_attack_code,
            "run": meta.run,
            "features": list(FEATURE_NAMES),
            "links": [
                {
                    "sender_pseudo": link.sender_pseudo,
                    "sender": link.sender,
                    "message_id": link.message_id.tolist(),
                    "rcv_time": np.round(link.rcv_time, decimals).tolist(),
                    "x": _with_nulls(x),
                }
                for link, x in zip(links, rounded)
            ],
            "log_dtau_first_message": "null (no earlier message from the same pseudonym)",
        }
        self._write(meta, payload)
        return {
            "run": meta.run,
            "family": meta.family,
            "group": meta.group,
            "time_window": meta.time_window,
            "file": meta.path.name,
            "receiver": meta.receiver,
            "receiver_pseudo": meta.receiver_pseudo,
            "receiver_attack_code": meta.receiver_attack_code,
            "n_type2": len(trace.own_gps),
            "n_type3": n_type3,
            "n_kept": n_kept,
            "n_duplicate_copies": n_type3 - len(np.unique(trace.heard_ids[:, 2])),
            "n_links": len(links),
            "n_pseudonym1_links": sum(link.sender_pseudo == 1 for link in links),
            "links": [[link.sender, link.sender_pseudo, len(x)] for link, x in zip(links, rounded)],
            "sums": self._partial_sums(links, rounded),
        }

    def _write(self, meta: TraceFile, payload: dict[str, Any]) -> None:
        target = self._out_root / meta.relative_path()
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("w") as handle:
            json.dump(payload, handle, separators=(",", ":"))

    @staticmethod
    def _partial_sums(links: list[Link], rounded: list[np.ndarray]) -> dict[int, list]:
        """Per sender vehicle: [count, sum, sum of squares], each a list over the 13 features.

        Counts are per feature because the first log_dtau of every link is missing (NaN) and left out.
        """
        sums: dict[int, list] = {}
        for link, x in zip(links, rounded):
            empty = (np.zeros(N_FEATURES), np.zeros(N_FEATURES), np.zeros(N_FEATURES))
            n, total, total_sq = sums.get(link.sender, empty)
            present = np.isfinite(x)
            values = np.where(present, x, 0.0)
            sums[link.sender] = [
                n + present.sum(axis=0),
                total + values.sum(axis=0),
                total_sq + (values * values).sum(axis=0),
            ]
        return {
            sender: [n.tolist(), total.tolist(), total_sq.tolist()] for sender, (n, total, total_sq) in sums.items()
        }


def _with_nulls(x: np.ndarray) -> list[list[float | None]]:
    """``x`` as nested lists, with NaN written as ``None`` (JSON ``null``)."""
    rows = x.tolist()
    for row in rows:
        if row[-1] != row[-1]:  # NaN is the only value not equal to itself
            row[-1] = None
    return rows

## Worker for the process pool

The pool uses the `fork` start method, so each worker inherits `SETTINGS` and the classes above. The worker
must be a top-level function so the pool can call it by name.

In [7]:
def prepare_trace_file(path: str) -> dict[str, Any]:
    """Prepares one raw trace file with the notebook's ``SETTINGS`` and returns its summary."""
    return TraceFilePreparer(SETTINGS).prepare(path)

## Run over every trace file

This finds every `traceJSON-*.json` file (the `traceGroundTruthJSON-*` files are not read), prepares them in
parallel, and collects one summary per file into `file_results` (sorted by run and file name, so the result
does not depend on which worker finished first). `source_totals` adds up the raw counts.

In [8]:
raw_files = sorted(str(p) for p in pathlib.Path(SETTINGS.raw_dir).glob("*_*/VeReMi_*/traceJSON-*.json"))
print(f"{len(raw_files):,} raw trace files under {SETTINGS.raw_dir}")
pathlib.Path(SETTINGS.out_dir).mkdir(parents=True, exist_ok=True)

started = time.perf_counter()
with multiprocessing.get_context("fork").Pool(SETTINGS.workers) as pool:
    file_results = list(pool.imap_unordered(prepare_trace_file, raw_files, chunksize=16))
file_results.sort(key=lambda r: (r["run"], r["file"]))
elapsed = time.perf_counter() - started

source_totals = {
    "files": len(file_results),
    "type2_rows": sum(r["n_type2"] for r in file_results),
    "type3_rows": sum(r["n_type3"] for r in file_results),
    "kept_messages": sum(r["n_kept"] for r in file_results),
    "duplicate_copies": sum(r["n_duplicate_copies"] for r in file_results),
    "links": sum(r["n_links"] for r in file_results),
    "pseudonym1_links": sum(r["n_pseudonym1_links"] for r in file_results),
}
assert source_totals["files"] == len(raw_files)
assert source_totals["kept_messages"] == source_totals["type3_rows"], "every received message must be kept"
print(f"prepared {source_totals['files']:,} files in {elapsed / 60:.1f} min")
for key, value in source_totals.items():
    print(f"  {key:24s} {value:>12,}")

23,032 raw trace files under data/VeReMi-Dataset
prepared 23,032 files in 1.0 min
  files                          23,032
  type2_rows                  2,362,697
  type3_rows                 20,364,658
  kept_messages              20,364,658
  duplicate_copies                    0
  links                       5,321,657
  pseudonym1_links               42,864


## The index: labels, split and normalisation for the whole prepared tree

Every raw trace file now has its prepared twin. What is still missing is one small file at the
root of the prepared tree, `index.json`, that answers the questions every later notebook asks:

- **Which vehicles are attackers?** The attack code of a vehicle is written in the name of its
  own trace file (`...-A16-...` means GridSybil). A sender we hear in one time window may have its
  file in another time window of the same scenario folder, so all time windows are searched.
  Senders that have no file anywhere get the code **-1** (unknown) and are counted.
- **Which vehicles are for training, validation and testing?** The four attack scenarios of one
  group (0709 or 1416) replay the *same* traffic, so the same car appears in all four folders.
  If we split per folder, a test car's near-identical copy sits in training and the scores look
  better than they are. So we split whole **vehicles** (key `"<group>:<sender>"`), pooled over the
  four scenarios, and check that no vehicle ends up in two splits.
- **How do we scale the 13 features?** Mean and standard deviation per feature, computed only
  from messages whose sender is a training vehicle (validation and test never influence it).
- **How much data is there?** Files, links and messages per scenario folder and class, and how
  many links are long enough for a 64- or 128-message sample.
- **How was it made?** Git commit, library versions and the time of creation.

This part only reads the small per-file summaries (`file_results`) returned by the workers
above; no prepared file is read twice.

In [9]:
from __future__ import annotations

import dataclasses
import datetime
import json
import math
import os
import pathlib
import platform
import re
import subprocess
import sys
from collections import defaultdict
from collections.abc import Iterable, Mapping, Sequence
from typing import Any

import numpy as np
import pandas as pd

CLASS_NAMES: dict[int, str] = {
    -1: "Unknown",
    0: "Benign",
    16: "GridSybil",
    17: "DataReplaySybil",
    18: "DoSRandomSybil",
    19: "DoSDisruptiveSybil",
}
SPLIT_NAMES: tuple[str, ...] = ("train", "pretrain_val", "val", "test")
UNKNOWN_CODE = -1
TRACE_FILE = re.compile(r"^traceJSON-(\d+)-(\d+)-A(\d+)-(\d+)-(\d+)\.json$")
INDEX_FEATURES: list[str] = list(
    getattr(SETTINGS, "features", None)
    or [
        "claimed_pos_x",
        "claimed_pos_y",
        "rx_pos_x",
        "rx_pos_y",
        "claimed_vel_x",
        "claimed_vel_y",
        "rx_vel_x",
        "rx_vel_y",
        "claimed_acl_x",
        "claimed_acl_y",
        "range",
        "bearing",
        "log_dtau",
    ]
)


class LeakageError(RuntimeError):
    """Raised when a vehicle would sit in two splits."""


def scenario_folder(result: Mapping[str, Any]) -> str:
    """``"GridSybil_1416/VeReMi_..."`` -> ``"GridSybil_1416"``."""
    return str(result["run"]).split("/")[0]


def file_name(result: Mapping[str, Any]) -> str:
    """The trace file name of one per-file result."""
    return str(result.get("file") or result["file_name"])


def vehicle_key(group: str, vehicle: int) -> str:
    """The split key of a vehicle: its group and its id, e.g. ``"1416:10383"``."""
    return f"{group}:{int(vehicle)}"


def settings_dict(settings: Any) -> dict[str, Any]:
    """The settings object as a plain JSON-friendly dict."""
    raw = dataclasses.asdict(settings) if dataclasses.is_dataclass(settings) else vars(settings)
    return {k: (str(v) if isinstance(v, pathlib.Path) else v) for k, v in raw.items()}

### Labels from file names

`LabelBook` reads only file *names* (never file contents, never the ground-truth files). For each
scenario folder it collects `vehicle -> attack code` from every time-window folder inside it, so a
sender heard in one window is still labelled when its own file sits in a sibling window. A vehicle
that shows two different codes inside one scenario folder would mean the data is not what we think,
so that stops the notebook.

In [10]:
class LabelBook:
    """Vehicle id -> attack code per scenario folder, from trace file names.

    Args:
        raw_dir: The raw ``VeReMi-Dataset`` folder (read only).

    Raises:
        ValueError: If a vehicle has two attack codes inside one scenario folder.
    """

    def __init__(self, raw_dir: str | pathlib.Path) -> None:
        self._raw_dir = pathlib.Path(raw_dir)
        self._codes: dict[str, dict[int, int]] = {}
        self._unknown: dict[str, set[int]] = defaultdict(set)

    def codes_of(self, folder: str) -> dict[int, int]:
        """All vehicles with a trace file somewhere in ``folder`` (any time window)."""
        if folder not in self._codes:
            codes: dict[int, int] = {}
            for path in sorted((self._raw_dir / folder).glob("*/traceJSON-*.json")):
                match = TRACE_FILE.match(path.name)
                if not match:
                    continue
                vehicle, code = int(match[1]), int(match[3])
                if codes.setdefault(vehicle, code) != code:
                    raise ValueError(f"vehicle {vehicle} has codes {codes[vehicle]} and {code} " f"in {folder}")
            self._codes[folder] = codes
        return self._codes[folder]

    def code(self, folder: str, vehicle: int) -> int:
        """The attack code of one vehicle, or -1 (counted) if it has no file in the folder."""
        code = self.codes_of(folder).get(int(vehicle), UNKNOWN_CODE)
        if code == UNKNOWN_CODE:
            self._unknown[folder].add(int(vehicle))
        return code

    def as_json(self, folders: Iterable[str]) -> dict[str, dict[str, int]]:
        """``{folder: {vehicle: code}}``: every vehicle with a file, plus unknown senders as -1."""
        out = {}
        for folder in sorted(folders):
            merged = dict(self.codes_of(folder))
            merged.update({v: UNKNOWN_CODE for v in self._unknown.get(folder, ())})
            out[folder] = {str(v): c for v, c in sorted(merged.items())}
        return out

    def unknown_counts(self, folders: Iterable[str]) -> dict[str, int]:
        """Number of senders without any trace file, per scenario folder."""
        return {folder: len(self._unknown.get(folder, ())) for folder in sorted(folders)}

### Vehicle-grouped split

`VehicleSplitter` takes the sorted list of all sender keys, shuffles it with the fixed seed and
cuts it into train / validation / test (70 / 15 / 15 %). A further 10 % of the training vehicles
become `pretrain_val`, the label-free validation set used while pretraining; it is cut out of
the training block, so changing that fraction never moves a vehicle into validation or test.
Labels play no part in the assignment. The same seed and the same set of vehicles always give the
same split.

In [11]:
class VehicleSplitter:
    """Assigns every vehicle key to exactly one of train / pretrain_val / val / test.

    Args:
        train_frac: Share of vehicles for training (pretrain_val included).
        val_frac: Share of vehicles for validation.
        pretrain_val_frac: Share of the training vehicles moved to pretrain_val.
        seed: Seed of the shuffle.
    """

    def __init__(self, train_frac: float, val_frac: float, pretrain_val_frac: float, seed: int) -> None:
        self.train_frac = train_frac
        self.val_frac = val_frac
        self.pretrain_val_frac = pretrain_val_frac
        self.seed = seed

    def assign(self, keys: Iterable[str]) -> dict[str, list[str]]:
        """Returns split name -> sorted vehicle keys.

        Raises:
            ValueError: If there are too few vehicles for the fractions.
            LeakageError: If the result is not a clean partition of the keys.
        """
        vehicles = np.array(sorted(set(keys)))
        shuffled = vehicles[np.random.default_rng(self.seed).permutation(len(vehicles))]
        n_total = len(shuffled)
        n_train = int(round(self.train_frac * n_total))
        n_val = int(round(self.val_frac * n_total))
        n_pretrain_val = int(round(self.pretrain_val_frac * n_train))
        if n_train - n_pretrain_val < 1 or n_val < 1 or n_total - n_train - n_val < 1:
            raise ValueError(f"{n_total} vehicles are too few for the split fractions")
        cuts = {
            "pretrain_val": shuffled[:n_pretrain_val],
            "train": shuffled[n_pretrain_val:n_train],
            "val": shuffled[n_train : n_train + n_val],
            "test": shuffled[n_train + n_val :],
        }
        split = {name: sorted(cuts[name].tolist()) for name in SPLIT_NAMES}
        self.check(split, vehicles.tolist())
        return split

    @staticmethod
    def check(split: Mapping[str, Sequence[str]], all_keys: Iterable[str] | None = None) -> None:
        """Raises `LeakageError` if a key is in two splits or (if given) a key is missing."""
        seen: dict[str, str] = {}
        for name, keys in split.items():
            for key in keys:
                if key in seen:
                    raise LeakageError(f"vehicle {key} is in both {seen[key]} and {name}")
                seen[key] = name
        if all_keys is not None:
            missing = set(all_keys) - set(seen)
            if missing:
                raise LeakageError(f"{len(missing)} vehicles are in no split, " f"e.g. {sorted(missing)[:3]}")

    def as_json(self) -> dict[str, Any]:
        """The split settings, stored next to the split."""
        return {
            "key": "<group>:<sender vehicle>",
            "train_frac": self.train_frac,
            "val_frac": self.val_frac,
            "pretrain_val_frac": self.pretrain_val_frac,
            "seed": self.seed,
        }

### Feature scaling from training vehicles only

Each worker returned, per sender vehicle, the number of kept messages and the per-feature sum and
sum of squares. `NormalisationStats` adds these up for the training vehicles only and turns them
into a mean and a standard deviation per feature. (`bearing` is an angle; it is scaled like the
other features, as plain numbers in (-π, π].)

In [12]:
class NormalisationStats:
    """Per-feature mean / std from per-vehicle partial sums ``[n, sum, sumsq]`` (each a list over features).

    Counts are per feature: the first ``log_dtau`` of every link is missing and is left out of its feature only.

    Args:
        n_features: Number of features (13).
    """

    def __init__(self, n_features: int) -> None:
        self.n = np.zeros(n_features, dtype=np.float64)
        self.total = np.zeros(n_features, dtype=np.float64)
        self.total_sq = np.zeros(n_features, dtype=np.float64)
        self.n_vehicles: set[str] = set()

    def add(self, key: str, n: Sequence[float], sums: Sequence[float], sumsq: Sequence[float]) -> None:
        """Adds one vehicle's partial sums from one file."""
        self.n += np.asarray(n, dtype=np.float64)
        self.total += np.asarray(sums, dtype=np.float64)
        self.total_sq += np.asarray(sumsq, dtype=np.float64)
        self.n_vehicles.add(key)

    def as_json(self, feature_names: Sequence[str]) -> dict[str, Any]:
        """``{"features", "mean", "std", "n_values", "n_vehicles", "source"}``.

        Raises:
            ValueError: If no message was added, or a feature has zero spread.
        """
        if np.any(self.n == 0):
            raise ValueError("no training messages to compute the normalisation from")
        mean = self.total / self.n
        var = np.maximum(self.total_sq / self.n - mean**2, 0.0)
        std = np.sqrt(var)
        if np.any(std == 0) or not np.all(np.isfinite(std)):
            bad = [f for f, s in zip(feature_names, std) if not (s > 0 and math.isfinite(s))]
            raise ValueError(f"features with no spread in train: {bad}")
        return {
            "features": list(feature_names),
            "mean": mean.round(6).tolist(),
            "std": std.round(6).tolist(),
            "n_values": [int(v) for v in self.n],
            "n_vehicles": len(self.n_vehicles),
            "source": "messages whose sender vehicle is in the train split (pretrain_val, "
            "val and test excluded); population std",
        }

### How much data there is

`CountsTable` counts, per scenario folder and class: files (the class of the **receiver** that
wrote the file), and links / messages / links with at least 64 or 128 messages (the class of the
**sender**, which is what a detector has to recognise). It also counts the same per split.

In [13]:
class CountsTable:
    """Accumulates file / link / message counts per (scenario folder, class) and per split."""

    LENGTHS = (64, 128)

    def __init__(self) -> None:
        self._rows: dict[tuple[str, str], dict[str, int]] = defaultdict(self._empty)
        self._split_rows: dict[tuple[str, str], dict[str, int]] = defaultdict(self._empty)

    @classmethod
    def _empty(cls) -> dict[str, int]:
        return {"files": 0, "links": 0, "messages": 0, **{f"links_ge_{n}": 0 for n in cls.LENGTHS}}

    def add_file(self, folder: str, receiver_code: int) -> None:
        """Counts one prepared file under the receiver's class."""
        self._rows[(folder, CLASS_NAMES.get(receiver_code, str(receiver_code)))]["files"] += 1

    def add_link(self, folder: str, sender_code: int, n_messages: int, split: str) -> None:
        """Counts one link under the sender's class, by folder and by split."""
        name = CLASS_NAMES.get(sender_code, str(sender_code))
        for row in (self._rows[(folder, name)], self._split_rows[(split, name)]):
            row["links"] += 1
            row["messages"] += int(n_messages)
            for n in self.LENGTHS:
                row[f"links_ge_{n}"] += int(n_messages >= n)

    def by_folder(self) -> dict[str, dict[str, dict[str, int]]]:
        """``{folder: {class: counts}}``."""
        out: dict[str, dict[str, dict[str, int]]] = defaultdict(dict)
        for (folder, name), row in sorted(self._rows.items()):
            out[folder][name] = dict(row)
        return dict(out)

    def by_split(self) -> dict[str, dict[str, dict[str, int]]]:
        """``{split: {class: counts}}`` (files are not split, only links)."""
        out: dict[str, dict[str, dict[str, int]]] = {name: {} for name in SPLIT_NAMES}
        for (split, name), row in sorted(self._split_rows.items()):
            out[split][name] = {k: v for k, v in row.items() if k != "files"}
        return out

### Provenance and a guarded writer

`Provenance` records what is needed to reproduce the data: the git commit (and whether the working
tree had uncommitted changes), the Python and NumPy versions and the creation time. `IndexWriter`
writes only inside the prepared-data folder from the settings, and never inside the raw dataset;
it writes to a temporary file first and renames it, so a crash never leaves half an index.

In [14]:
class Provenance:
    """Code version and environment of this run.

    Args:
        repo_root: Folder inside the git repository.
    """

    def __init__(self, repo_root: str | pathlib.Path = ".") -> None:
        self._root = pathlib.Path(repo_root)

    def as_json(self) -> dict[str, Any]:
        """``{"git_commit", "git_dirty", "python", "numpy", "pandas", "platform", "created_utc"}``."""
        return {
            "git_commit": self._git("rev-parse", "HEAD"),
            "git_dirty": bool(self._git("status", "--porcelain")),
            "python": sys.version.split()[0],
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "platform": platform.platform(),
            "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        }

    def _git(self, *args: str) -> str:
        try:
            done = subprocess.run(["git", *args], cwd=self._root, capture_output=True, text=True, check=True)
        except (OSError, subprocess.CalledProcessError):
            return ""
        return done.stdout.strip()


class IndexWriter:
    """Writes JSON files, but only inside ``out_dir`` and never inside ``raw_dir``.

    Args:
        out_dir: The prepared-data folder (e.g. ``src/data/prepared_data``).
        raw_dir: The raw dataset folder, which must never be written.

    Raises:
        PermissionError: If ``out_dir`` lies inside ``raw_dir``.
    """

    def __init__(self, out_dir: str | pathlib.Path, raw_dir: str | pathlib.Path) -> None:
        self.out_dir = pathlib.Path(out_dir).resolve()
        self._raw_dir = pathlib.Path(raw_dir).resolve()
        if self.out_dir.is_relative_to(self._raw_dir):
            raise PermissionError(f"refusing to write inside the raw dataset: {self.out_dir}")

    def write(self, relative: str, payload: Mapping[str, Any]) -> pathlib.Path:
        """Writes ``payload`` to ``out_dir / relative`` and returns the path.

        Raises:
            PermissionError: If the target resolves outside ``out_dir``.
        """
        target = (self.out_dir / relative).resolve()
        if not target.is_relative_to(self.out_dir):
            raise PermissionError(f"refusing to write outside {self.out_dir}: {target}")
        target.parent.mkdir(parents=True, exist_ok=True)
        tmp = target.with_name(target.name + ".tmp")
        with tmp.open("w") as handle:
            json.dump(payload, handle, indent=1)
        os.replace(tmp, target)
        return target

### Building the index

`IndexBuilder` ties the pieces together: it labels every sender, splits the sender vehicles, adds
up the training sums, counts everything and returns the index as one dictionary.

In [15]:
class IndexBuilder:
    """Builds the content of ``index.json`` from the per-file results.

    Args:
        results: One summary dict per prepared file (returned by the workers).
        labels: Label lookup over the raw file names.
        splitter: The vehicle splitter.
        feature_names: The 13 feature names, in order.
    """

    def __init__(
        self,
        results: Sequence[Mapping[str, Any]],
        labels: LabelBook,
        splitter: VehicleSplitter,
        feature_names: Sequence[str],
    ) -> None:
        self._results = results
        self._labels = labels
        self._splitter = splitter
        self._features = list(feature_names)

    def sender_keys(self) -> set[str]:
        """Every ``"<group>:<sender>"`` heard in any link of any file."""
        return {vehicle_key(r["group"], link[0]) for r in self._results for link in r["links"]}

    def build(self, settings: Any, source_totals: Mapping[str, Any]) -> dict[str, Any]:
        """Returns the full index dictionary."""
        split = self._splitter.assign(self.sender_keys())
        split_of = {key: name for name, keys in split.items() for key in keys}
        counts = CountsTable()
        norm = NormalisationStats(len(self._features))
        folders = sorted({scenario_folder(r) for r in self._results})
        for result in self._results:
            folder, group = scenario_folder(result), str(result["group"])
            counts.add_file(folder, int(result["receiver_attack_code"]))
            for sender, _pseudo, n_messages in result["links"]:
                counts.add_link(
                    folder, self._labels.code(folder, sender), int(n_messages), split_of[vehicle_key(group, sender)]
                )
            for sender, (n, sums, sumsq) in result["sums"].items():
                key = vehicle_key(group, int(sender))
                if split_of[key] == "train":
                    norm.add(key, n, sums, sumsq)
        return {
            "features": self._features,
            "settings": settings_dict(settings),
            "source_totals": dict(source_totals),
            "labels": self._labels.as_json(folders),
            "labels_unknown": self._labels.unknown_counts(folders),
            "class_names": {str(k): v for k, v in CLASS_NAMES.items()},
            "split": {
                "settings": self._splitter.as_json(),
                "n_vehicles": {name: len(keys) for name, keys in split.items()},
                "vehicles": split,
                "check": "passed: no vehicle key in two splits",
            },
            "normalisation": norm.as_json(self._features),
            "counts": counts.by_folder(),
            "counts_by_split": counts.by_split(),
            "provenance": Provenance().as_json(),
        }

In [16]:
label_book = LabelBook(SETTINGS.raw_dir)
splitter = VehicleSplitter(SETTINGS.train_frac, SETTINGS.val_frac, SETTINGS.pretrain_val_frac, SETTINGS.seed)
index = IndexBuilder(file_results, label_book, splitter, INDEX_FEATURES).build(SETTINGS, source_totals)
index_path = IndexWriter(SETTINGS.out_dir, SETTINGS.raw_dir).write("index.json", index)
print(f"wrote {index_path} ({index_path.stat().st_size / 1e6:.1f} MB)")
print("vehicles per split:", index["split"]["n_vehicles"])
print("senders without a trace file:", {k: v for k, v in index["labels_unknown"].items() if v})

wrote /Users/bibekgupta/Downloads/projects/sybil-llm/src/data/prepared_data/index.json (0.5 MB)
vehicles per split: {'train': 4730, 'pretrain_val': 526, 'val': 1126, 'test': 1126}
senders without a trace file: {'GridSybil_0709': 24}


## Checks on what was written

Before anything is trained on this data, we check four things:

- **The tree mirrors the raw dataset**: exactly one prepared file for every raw `traceJSON-*`
  file, with the same scenario folder, time-window folder and file name (ground-truth files are
  not mirrored).
- **A random sample of prepared files is sound**: each loads, has the 13 feature names, every
  link has as many message ids and receive times as feature rows, every row is 13 finite numbers,
  receive times never go backwards, and the number of rows matches what the worker reported.
- **No vehicle is in two splits**, read back from the `index.json` on disk, and every sender
  heard anywhere has a split.
- **A summary table** of what we have, per scenario folder and class, and per split.

In [17]:
class PreparedTreeCheck:
    """Checks the prepared tree against the raw tree and the index.

    Args:
        raw_dir: The raw dataset folder.
        out_dir: The prepared-data folder.
        results: The per-file results returned by the workers.
        sample_size: How many prepared files to open and inspect.
        seed: Seed for picking the sample.
    """

    def __init__(
        self,
        raw_dir: str | pathlib.Path,
        out_dir: str | pathlib.Path,
        results: Sequence[Mapping[str, Any]],
        sample_size: int = 50,
        seed: int = 42,
    ) -> None:
        self._raw_dir = pathlib.Path(raw_dir)
        self._out_dir = pathlib.Path(out_dir)
        self._results = {(str(r["run"]), file_name(r)): r for r in results}
        self._sample_size = sample_size
        self._seed = seed

    @staticmethod
    def _trace_files(root: pathlib.Path) -> set[str]:
        return {p.relative_to(root).as_posix() for p in root.glob("*/*/traceJSON-*.json")}

    def mirror(self) -> int:
        """Returns the number of files; raises AssertionError if the trees differ."""
        raw, prepared = self._trace_files(self._raw_dir), self._trace_files(self._out_dir)
        missing, extra = sorted(raw - prepared), sorted(prepared - raw)
        assert not missing and not extra, (
            f"{len(missing)} raw files without a prepared twin (e.g. {missing[:3]}), "
            f"{len(extra)} prepared files without a raw source (e.g. {extra[:3]})"
        )
        return len(raw)

    def sample(self) -> int:
        """Inspects a random sample of prepared files; returns the number of rows checked."""
        files = sorted(self._trace_files(self._out_dir))
        rng = np.random.default_rng(self._seed)
        picked = rng.choice(len(files), size=min(self._sample_size, len(files)), replace=False)
        n_rows = 0
        for i in sorted(picked):
            rel = files[i]
            data = json.loads((self._out_dir / rel).read_text())
            assert data["features"] == INDEX_FEATURES, f"{rel}: wrong feature list"
            rows_in_file = 0
            for link in data["links"]:
                x = np.asarray(link["x"], dtype=np.float64).reshape(-1, len(INDEX_FEATURES))  # null -> NaN
                assert (
                    len(x) == len(link["x"]) == len(link["message_id"]) == len(link["rcv_time"])
                ), f"{rel}: link {link['sender_pseudo']} has ragged columns"
                assert (
                    np.isfinite(x[:, :-1]).all() and np.isfinite(x[1:, -1]).all()
                ), f"{rel}: missing or non-finite feature value"
                assert np.isnan(x[0, -1]), f"{rel}: first log_dtau of a link should be null"
                assert np.all(np.diff(link["rcv_time"]) >= 0), f"{rel}: receive times go back"
                rows_in_file += len(x)
            run, name = rel.rsplit("/", 1)
            reported = self._results[(run, name)]["n_kept"]
            assert rows_in_file == reported, f"{rel}: {rows_in_file} rows, worker said {reported}"
            n_rows += rows_in_file
        return n_rows

    def split(self, index_file: pathlib.Path) -> dict[str, int]:
        """Re-reads the split from disk and checks it; returns vehicles per split."""
        split = json.loads(index_file.read_text())["split"]["vehicles"]
        heard = {vehicle_key(r["group"], link[0]) for r in self._results.values() for link in r["links"]}
        VehicleSplitter.check(split, heard)
        return {name: len(keys) for name, keys in split.items()}


checker = PreparedTreeCheck(SETTINGS.raw_dir, SETTINGS.out_dir, file_results, seed=SETTINGS.seed)
print(f"mirror: {checker.mirror():,} prepared files, one per raw trace file")
print(f"sample: {checker.sample():,} feature rows in the sampled files are 13 finite numbers")
print(f"split:  no vehicle in two splits {checker.split(index_path)}")

mirror: 23,032 prepared files, one per raw trace file
sample: 39,440 feature rows in the sampled files are 13 finite numbers
split:  no vehicle in two splits {'train': 4730, 'pretrain_val': 526, 'val': 1126, 'test': 1126}


In [18]:
summary = pd.DataFrame(
    [
        {"scenario folder": folder, "class": name, **row}
        for folder, classes in index["counts"].items()
        for name, row in classes.items()
    ]
)
total = summary.drop(columns=["scenario folder", "class"]).sum()
summary.loc[len(summary)] = {"scenario folder": "all", "class": "", **total.to_dict()}
print("Per scenario folder and class (files by receiver class; links and messages by sender class)")
print(summary.to_string(index=False))

by_split = pd.DataFrame(
    [
        {"split": split, "class": name, **row}
        for split, classes in index["counts_by_split"].items()
        for name, row in classes.items()
    ]
)
print("\nPer split and sender class")
print(by_split.to_string(index=False))

Per scenario folder and class (files by receiver class; links and messages by sender class)
        scenario folder              class  files   links  messages  links_ge_64  links_ge_128
   DataReplaySybil_0709             Benign   2847  103860   2081685         4865           603
   DataReplaySybil_0709    DataReplaySybil   1221  588828    873068            0             0
   DataReplaySybil_1416             Benign   1184   21658    528518         1856           317
   DataReplaySybil_1416    DataReplaySybil    507  143829    239440            1             0
DoSDisruptiveSybil_0709             Benign   2847  103859   2081446         4873           600
DoSDisruptiveSybil_0709 DoSDisruptiveSybil   1221 1547646   1741441            0             0
DoSDisruptiveSybil_1416             Benign   1184   21688    528397         1853           315
DoSDisruptiveSybil_1416 DoSDisruptiveSybil    507  381293    477450            0             0
    DoSRandomSybil_0709             Benign   2847  10